# 01 — Adatbetöltés és feltáró adatelemzés (EDA)

**Szakdolgozat:** Sentiment Analysis of Customer Reviews using Machine Learning Techniques
**Témavezető:** Mutaz AlShafeey

---

## Mit csinálunk ebben a notebookban?

| Lépés | Mit | Melyik dolgozat-fejezetbe megy |
|---|---|---|
| 1 | Környezet ellenőrzése | — |
| 2 | Adat betöltése | 4. Data Sources |
| 3 | Első ránézés, adatminőség | 6.1 Preprocessing |
| 4 | Sentiment címkék képzése a pontszámból | 6.1 Preprocessing |
| 5 | Szűrés és tisztítás | 6.1 Preprocessing |
| 6 | Feltáró elemzés + ábrák | 9. Exploratory Data Analysis |
| 7 | Aspektus-előzetes | 7. ABSA / 9. EDA |
| 8 | 60/20/20 felosztás és mentés | 8. Data Split |

> **Tipp a dolgozathoz:** minden ábrát és táblázatot, amit hasznosnak találsz,
> mentsd el a `results/` mappába (a kód meg is teszi). Így három hónap múlva
> nem kell újrafuttatnod mindent, amikor az ábrát be akarod illeszteni.

## 1. Környezet és importok

In [ ]:
# Ha a notebooks/ mappából futtatod, a szülőmappát hozzá kell adni az elérési úthoz,
# hogy a `src` csomagot importálni tudja a Python.
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import set_seed
from src.config import CFG, PATHS, DATASETS
from src import data_loading as dl
from src import preprocessing as pp
from src import aspects as asp
from src import evaluation as ev

# Reprodukálhatóság: EZ A LEGFONTOSABB SOR a notebookban.
set_seed()
PATHS.ensure()

# Egységes ábrastílus az egész dolgozathoz
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 11

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

print(f"pandas {pd.__version__} | numpy {np.__version__}")
print(f"Seed: {CFG.random_seed}")
print(f"Projekt gyökér: {PATHS.root}")

## 2. Adat betöltése

**Fő adathalmaz: Booking.com Accommodation Reviews (Hugging Face).**

Miért ez? Hivatalosan a Booking.com publikálta, **CC BY-SA 4.0** licenc alatt (nem
kereskedelmi célra), ami egy szakdolgozatnál tiszta jogi helyzetet jelent — ezt a
dolgozat 4. fejezetében hivatkozni is tudod. Ráadásul két sorból betölthető, tehát
a munkád **reprodukálható**: a bíráló ugyanezt a kódot lefuttatva ugyanazt az adatot kapja.

**Most csak egy mintát töltünk be.** 1,6 millió soron fejleszteni időpocsékolás:
minden hiba után percekig vársz. A `dev_sample_size` a `config.py`-ban állítható,
és a *végleges* futtatásnál kapcsolod ki (`None`).

In [ ]:
# FEJLESZTÉSI MÓD: kis minta, gyors iteráció.
# A végleges futtatáshoz írd át N_ROWS = None -ra (és számolj 15-30 perccel).
N_ROWS = CFG.dev_sample_size   # = 50_000

df_raw = dl.load_dataset_by_name("booking_hf", n_rows=N_ROWS)

print(f"Betöltve: {len(df_raw):,} sor, {df_raw.shape[1]} oszlop")
df_raw.head(3)

> **Ha a Hugging Face letöltés elakad** (a dataset viewer néha hibát jelez),
> két megoldás van:
> 1. `pip install -U datasets huggingface-hub` és próbáld újra
> 2. Töltsd le kézzel a CSV-t a [GitHub repóból](https://github.com/bookingcom/ml-dataset-reviews),
>    tedd a `data/raw/` mappába, és a loader megtalálja.
>
> Alternatívaként a `kaggle_515k` adathalmazzal is dolgozhatsz ugyanezzel a kóddal:
> `dl.load_dataset_by_name("kaggle_515k", n_rows=50_000)`

## 3. Első ránézés — adatminőség

In [ ]:
# Alapstatisztikák. Mindig ezzel kezdj: mennyi a hiányzó érték, mekkorák a szövegek?
print("=== Oszlopok és típusok ===")
print(df_raw.dtypes.to_string())

print("\n=== Hiányzó értékek ===")
missing = df_raw.isna().sum()
print(missing[missing > 0].to_string() if missing.sum() else "Nincs hiányzó érték.")

print("\n=== Duplikátumok (szöveg alapján) ===")
n_dup = df_raw["text"].duplicated().sum()
print(f"{n_dup:,} db ({n_dup/len(df_raw):.2%})")

print("\n=== Pontszám eloszlása ===")
print(df_raw["score"].describe().to_string())

In [ ]:
# Szöveghossz — ez dönti el, mekkora max_length kell majd a BERT-nek.
df_raw["n_chars"] = df_raw["text"].str.len()
df_raw["n_words"] = df_raw["text"].str.split().str.len()

print(df_raw[["n_chars", "n_words"]].describe(
    percentiles=[.5, .75, .9, .95, .99]).round(1).to_string())

q95 = df_raw["n_words"].quantile(0.95)
print(f"\nA reviewk 95%-a rövidebb, mint {q95:.0f} szó.")
print(f"=> A CFG.max_length = {CFG.max_length} token ebből kb. "
      f"{'ELÉG' if CFG.max_length > q95 * 1.3 else 'KEVÉS LEHET — érdemes növelni'}.")

In [ ]:
# Nézz meg néhány valódi példát — mindig, minden adathalmaznál!
# Sok hibát csak így veszel észre (placeholder szövegek, kódolási hibák, spam).
for i, r in df_raw.sample(5, random_state=CFG.random_seed).iterrows():
    print(f"--- score: {r['score']} ---")
    print(r["text"][:350])
    print()

## 4. Sentiment címkék képzése a pontszámból

**Ez a dolgozat egyik legfontosabb módszertani döntése, és meg kell indokolni.**

Az adathalmazban nincs kész `positive / neutral / negative` címke, csak egy 1–10-es
vendégpontszám. Ebből képezzük a három osztályt:

| Feltétel | Címke |
|---|---|
| `score <= 6` | negative |
| `6 < score < 8` | neutral |
| `score >= 8` | positive |

**Amit a dolgozatba írj:** ez nem egy „természetes törvény", hanem egy döntés.
A szakirodalomban gyakori a 6/8 vágás, de a küszöb önkényes. Ezért futtatunk
alább egy **érzékenységi vizsgálatot** is: megnézzük, mennyire változik az
osztályeloszlás más küszöbökkel. Ez pont az a fajta módszertani alaposság,
amit a bírálók keresnek.

In [ ]:
df = pp.add_sentiment_labels(df_raw)

dist = df["sentiment"].value_counts().reindex(list(CFG.labels))
print("Osztályeloszlás a 6/8 küszöbbel:")
print(pd.DataFrame({"db": dist, "arány": (dist / len(df)).round(4)}).to_string())

imbalance = dist.max() / dist.min()
print(f"\nKiegyensúlyozatlansági arány: {imbalance:.1f}x")
print("=> Ezért használunk MACRO-F1-et fő metrikaként és class_weight='balanced'-et.")

In [ ]:
# ÉRZÉKENYSÉGI VIZSGÁLAT — ez a táblázat mehet a dolgozatba.
rows = []
for neg_t, pos_t in [(5.0, 8.0), (6.0, 8.0), (6.0, 8.5), (7.0, 9.0), (6.5, 8.5)]:
    labels = [pp.score_to_sentiment(s, sm, neg_threshold=neg_t, pos_threshold=pos_t)
              for s, sm in zip(df_raw["score"], df_raw["score_scale_max"])]
    vc = pd.Series(labels).value_counts(normalize=True)
    rows.append({
        "küszöbök": f"neg<={neg_t} / pos>={pos_t}",
        "negative": vc.get("negative", 0),
        "neutral":  vc.get("neutral", 0),
        "positive": vc.get("positive", 0),
    })

sens = pd.DataFrame(rows).set_index("küszöbök").round(3)
display(sens)
ev.save_table(sens, "01_label_threshold_sensitivity")
print("\nMegfigyelés: a neutral osztály mérete erősen függ a küszöbtől — "
      "ezt a dolgozatban külön érdemes tárgyalni.")

## 5. Szűrés és tisztítás

In [ ]:
# A build_dataset() elvégzi: szűrés -> címkézés -> kétféle tisztítás.
# Az első futtatás 50k soron kb. 1-3 perc (a lemmatizálás a lassú rész).
df = pp.build_dataset(df_raw, classic_clean=True, minimal_clean=True, balance="none")

print(f"\nVégleges méret: {len(df):,} sor")
df[["text", "text_clean", "score", "sentiment"]].head(3)

In [ ]:
# Nézd meg, mit CSINÁLT a tisztítás egy konkrét példán.
# Ezt tedd be a dolgozatba illusztrációként (6.1 fejezet)!
ex = df.iloc[0]
print("EREDETI:\n", ex["text"][:300])
print("\nBERT-hez (minimális tisztítás):\n", ex["text_bert"][:300])
print("\nTF-IDF-hez (teljes tisztítás):\n", ex["text_clean"][:300])

## 6. Feltáró adatelemzés (EDA) — a dolgozat 9. fejezete

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# (a) Pontszám-eloszlás a küszöbökkel
axes[0].hist(df["score"], bins=40, color="#4a7ba7", edgecolor="white")
axes[0].axvline(CFG.neg_threshold, color="#c1483b", ls="--", lw=2, label=f"neg ≤ {CFG.neg_threshold}")
axes[0].axvline(CFG.pos_threshold, color="#3d7a5a", ls="--", lw=2, label=f"pos ≥ {CFG.pos_threshold}")
axes[0].set(xlabel="Vendégpontszám", ylabel="Reviewk száma", title="(a) Pontszám-eloszlás")
axes[0].legend(frameon=False, fontsize=9)

# (b) Osztályeloszlás
order = list(CFG.labels)
counts = df["sentiment"].value_counts().reindex(order)
axes[1].bar(order, counts, color=["#c1483b", "#b8b4ab", "#3d7a5a"])
for i, v in enumerate(counts):
    axes[1].text(i, v, f"{v:,}\n({v/len(df):.1%})", ha="center", va="bottom", fontsize=9)
axes[1].set(ylabel="Reviewk száma", title="(b) Sentiment-osztályok")
axes[1].margins(y=0.18)

# (c) Szöveghossz osztályonként
sns.boxplot(data=df, x="sentiment", y="n_words", order=order, ax=axes[2],
            palette=["#c1483b", "#b8b4ab", "#3d7a5a"], showfliers=False)
axes[2].set(xlabel="", ylabel="Szavak száma", title="(c) Reviewhossz osztályonként")

plt.tight_layout()
ev.save_figure("01_eda_overview")
plt.show()

In [ ]:
# Megfigyelés a (c) ábrához — írd le a dolgozatban, amit LÁTSZ:
print(df.groupby("sentiment", observed=True)["n_words"].agg(
    ["count", "mean", "median"]).round(1).to_string())
print("\nTipikus megfigyelés: a negatív reviewk HOSSZABBAK — az elégedetlen "
      "vendégek részletesen leírják a problémát, míg a pozitív értékelés gyakran "
      "csak pár szó. Ez önmagában is prediktív jel, és a hibaelemzésnél visszatér.")

In [ ]:
# Leggyakoribb szavak osztályonként — az első betekintés abba, MI hajtja a sentimentet.
from sklearn.feature_extraction.text import CountVectorizer

fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
colors = {"negative": "#c1483b", "neutral": "#8a8680", "positive": "#3d7a5a"}

for ax, label in zip(axes, CFG.labels):
    texts = df.loc[df["sentiment"].astype(str) == label, "text_clean"]
    if len(texts) == 0:
        continue
    cv = CountVectorizer(max_features=15, ngram_range=(1, 2), min_df=2)
    X = cv.fit_transform(texts)
    freq = pd.Series(np.asarray(X.sum(axis=0)).ravel(),
                     index=cv.get_feature_names_out()).sort_values()
    ax.barh(freq.index, freq.values, color=colors[label])
    ax.set_title(f"{label} (n={len(texts):,})")
    ax.set_xlabel("Előfordulás")

plt.tight_layout()
ev.save_figure("01_top_words_by_sentiment")
plt.show()

## 7. Aspektus-előzetes — az ABSA felvezetése

Itt a **kulcsszó-alapú** aspektusdetektálást futtatjuk le. Ez lesz a **baseline**,
amihez a későbbi transformer-alapú ABSA-t hasonlítod (RQ4).

Fontos módszertani megjegyzés a dolgozatba: itt a **review-szintű** sentimentet
vetítjük rá a mondatokra, ami közelítés. Egy pozitív összpontszámú review is
tartalmazhat negatív mondatot — pont ez a dolgozat kiinduló példája. A pontos
mondat-szintű címkékhez kell a **HRAST** adathalmaz (04-es notebook).

In [ ]:
# Mondatra bontás + aspektusok detektálása.
# Fejlesztéshez elég egy minta, mert ez lassú művelet.
sample = df.sample(min(20_000, len(df)), random_state=CFG.random_seed)

sent_df = asp.explode_to_sentences(sample, text_col="text")
sent_df = asp.add_aspect_columns(sent_df, text_col="sentence")

print(f"{len(sample):,} review -> {len(sent_df):,} mondat")
print(f"Átlagosan {sent_df['n_aspects'].mean():.2f} aspektus / mondat")
print(f"A mondatok {(sent_df['n_aspects'] == 0).mean():.1%}-ában nem talált aspektust "
      f"-> ezeket a transformer-alapú módszer részben meg fogja találni (implicit aspektusok).")

In [ ]:
# RQ3 KÖZVETLEN VÁLASZA: mely aspektusok kötődnek a negatív sentimenthez?
aspect_table = asp.aspect_sentiment_table(sent_df, min_count=50)
display(aspect_table)
ev.save_table(aspect_table, "01_aspect_sentiment_distribution")

print("\nOlvasat:")
print("  neg_lift > 1  -> ez az aspektus PANASZHAJTÓ (az átlagnál gyakrabban negatív)")
print("  neg_lift < 1  -> ez az aspektus ELÉGEDETTSÉGHAJTÓ")

In [ ]:
asp.plot_aspect_sentiment(aspect_table, save_as="01_aspect_sentiment_stacked")
plt.show()

In [ ]:
# Konkrét példák: hogyan néz ki a "textuális bizonyíték" kiemelése?
# Ez a dolgozat 10. fejezetének 3. szintje ("Evidence / associated factor").
neg_sent = sent_df[(sent_df["sentiment"].astype(str) == "negative") &
                   (sent_df["n_aspects"] > 0)].head(6)

for _, r in neg_sent.iterrows():
    print(f"MONDAT: {r['sentence'][:150]}")
    for a in r["aspects"]:
        spans = asp.extract_evidence(r["sentence"], a)
        if spans:
            neg = "  [TAGADÁS!]" if asp.has_negation(spans[0]) else ""
            print(f"   • {a:16s} -> \"{spans[0]}\"{neg}")
    print()

> **Írd fel magadnak a dolgozathoz:** figyeld meg, hol bukik el a kulcsszavas
> módszer. Tipikus esetek: tagadás („the room was **not** clean"), irónia,
> implicit aspektus („we could not sleep" = zaj, kulcsszó nélkül), és többszörös
> aspektus egy mondatban. Ez a lista lesz az indoklásod arra, **miért kell**
> transformer-alapú ABSA — és ez egy önálló, értékes dolgozat-alfejezet.

## 8. Felosztás 60/20/20 és mentés

In [ ]:
train_df, val_df, test_df = pp.train_val_test_split(df)

# Ellenőrzés: tényleg megmaradt az osztályarány mindhárom halmazban?
check = pd.DataFrame({
    "train": train_df["sentiment"].value_counts(normalize=True),
    "val":   val_df["sentiment"].value_counts(normalize=True),
    "test":  test_df["sentiment"].value_counts(normalize=True),
}).reindex(list(CFG.labels)).round(4)
display(check)
print("A három oszlopnak szinte azonosnak kell lennie -> a rétegzés működik.")

In [ ]:
# Mentés parquet formátumban (gyorsabb és kisebb, mint a CSV).
# A 02-es és 03-as notebook innen tölt be -> nem kell újra előfeldolgozni.
PATHS.ensure()
keep = ["review_id", "text", "text_clean", "text_bert", "score", "sentiment", "dataset"]

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    path = PATHS.processed / f"booking_{name}.parquet"
    part[keep].to_parquet(path, index=False)
    print(f"{name:6s} -> {path.name:28s} ({len(part):,} sor)")

# A konfigurációt is mentsük az adat mellé: így pontosan tudni fogod,
# milyen beállításokkal készült ez a splitelt adathalmaz.
CFG.to_json(PATHS.processed / "config_snapshot.json")
print("\nKonfiguráció elmentve: config_snapshot.json")

---

## Összefoglaló — mi készült el, és mi megy a dolgozatba

**Elmentett ábrák** (`results/figures/`):
- `01_eda_overview.png` — pontszám-eloszlás, osztályeloszlás, hosszeloszlás → **9. fejezet**
- `01_top_words_by_sentiment.png` — leggyakoribb szavak osztályonként → **9. fejezet**
- `01_aspect_sentiment_stacked.png` — aspektusonkénti sentiment-megoszlás → **7. / 9. fejezet**

**Elmentett táblázatok** (`results/tables/`):
- `01_label_threshold_sensitivity.csv` — küszöb-érzékenységi vizsgálat → **6.1 fejezet**
- `01_aspect_sentiment_distribution.csv` — aspektus × sentiment → **RQ3 válasza**

**Írj meg MOST, amíg friss** (nyiss egy `docs/naplo.md` fájlt):
1. Miért ezt az adathalmazt választottad, és mi a licenc?
2. Miért 6/8 a küszöb, és mit mutatott az érzékenységi vizsgálat?
3. Mit dobott ki a szűrés, és miért (a szűrési napló számai)?
4. Mit vettél észre az EDA-ban? (hosszkülönbség, kiegyensúlyozatlanság)
5. Hol bukik el a kulcsszavas aspektusdetektálás?

> Ez a napló lesz a dolgozat 6., 7. és 9. fejezetének a nyersanyaga.
> Sokkal könnyebb most leírni, mint három hónap múlva visszaemlékezni rá.

**Következő:** `02_classical_baseline.ipynb` — TF-IDF + Naive Bayes / LogReg / SVM (RQ1, RQ2).